# 07 — RetinaNet: Controlled Tuning & 3-fold verification (Colab)

**Goal:** tune the same RetinaNet ResNet-50 FPN v2 baseline without changing its original results. Keep the exact video-level `cv_splits_step6.csv`, 6 epochs, batch=2, seed=42, COCOeval with ignored `iscrowd` areas. Train candidates **on fold 0 only**, then validate selected configuration on folds 1 and 2.

**Please do not run all cells automatically.** Explicit `run_trial(...)` calls are provided below, initially commented. Each run saves its own latest checkpoint and metrics to Drive and resumes when interrupted. This notebook **does not guarantee** a Kaggle leaderboard improvement; it measures controlled CV changes.

Training annotations marked `iscrowd=1` are skipped, but preserved as ignore regions in evaluation. We reuse the original baseline's behavior. Model logits are indexed 0–7, COCO categories 1–8.

## 1 — Setup & Google Drive
Run on a GPU runtime. Reuse the existing Drive ZIP, not Kaggle API.

In [ ]:
%pip -q install pycocotools

import gc
import json
import math
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from collections import defaultdict
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from google.colab import drive
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

drive.mount('/content/drive')

SEED = 42
FRAME_STEP = 6
N_FOLDS = 3
BATCH_SIZE = 2
EPOCHS = 6
MIN_SIZE = 640
MAX_SIZE = 1024
NUM_WORKERS = 0  # avoids Colab multiprocessing teardown errors

ROOT = Path('/content/drive/MyDrive/UCU_AI_HW2')
ARCHIVE_PATH = ROOT / 'visdrone_competition.zip'
SPLIT_PATH = ROOT / 'cv_splits_step6.csv'
RESULTS_ROOT = ROOT / 'retinanet_v2_tuning'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Select a GPU runtime (Runtime > Change runtime type).'
DEVICE = torch.device('cuda')
assert ARCHIVE_PATH.is_file(), f'Missing existing dataset archive: {ARCHIVE_PATH}'
assert SPLIT_PATH.is_file(), f'Missing exact Faster R-CNN folds: {SPLIT_PATH}'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset archive:', ARCHIVE_PATH)
print('CV split:', SPLIT_PATH)
print('Tuning root:', RESULTS_ROOT)
BASELINE_ROOT = ROOT / 'retinanet_v2_baseline'
assert (BASELINE_ROOT / 'fold_0' / 'validation_metrics.csv').is_file(), 'Missing original RetinaNet baseline on Drive.'

Mounted at /content/drive
Torch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
GPU: Tesla T4
Dataset archive: /content/drive/MyDrive/UCU_AI_HW2/visdrone_competition.zip
CV split: /content/drive/MyDrive/UCU_AI_HW2/cv_splits_step6.csv
Tuning root: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_tuning


## 2 — Use the exact local dataset, original annotations and folds
Do not regenerate CV folds.

In [ ]:
LOCAL_DIR = Path('/content/visdrone')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)


def find_root(base):
    for ann_file in base.rglob('train/annotations_coco.json'):
        root = ann_file.parent.parent
        if (root / 'train/images').is_dir() and (root / 'val/annotations_coco.json').is_file():
            return root
    return None

DATA_ROOT = find_root(LOCAL_DIR)

if DATA_ROOT is None:
    assert zipfile.is_zipfile(ARCHIVE_PATH), 'Drive ZIP is invalid/incomplete.'
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        members = archive.infolist()
        required = sum(m.file_size for m in members)
        available = shutil.disk_usage('/content').free
        print(f'Uncompressed size: {required/1e9:.2f} GB; free: {available/1e9:.2f} GB')
        assert available > required * 1.10, 'Insufficient free local disk for the dataset.'
        # ZIP is an input from the competition: also reject paths escaping the extraction root.
        base = LOCAL_DIR.resolve()
        for member in members:
            destination = (LOCAL_DIR / member.filename).resolve()
            assert destination == base or base in destination.parents, f'Unsafe ZIP member: {member.filename}'
        print('Extracting dataset locally...')
        archive.extractall(LOCAL_DIR)
    DATA_ROOT = find_root(LOCAL_DIR)

assert DATA_ROOT is not None, 'Expected train/annotations_coco.json + train/images not found.'

with (DATA_ROOT / 'train/annotations_coco.json').open('r') as f:
    train_coco = json.load(f)
with (DATA_ROOT / 'val/annotations_coco.json').open('r') as f:
    holdout_coco = json.load(f)

assert len(train_coco['images']) == 24201, 'Unexpected train image count.'
assert len(holdout_coco['images']) == 2846, 'Unexpected untouched holdout size.'
print('Dataset root:', DATA_ROOT)
print('Train images:', len(train_coco['images']))
print('Original holdout images:', len(holdout_coco['images']))

Uncompressed size: 10.98 GB; free: 120.80 GB
Extracting dataset locally...
Dataset root: /content/visdrone
Train images: 24201
Original holdout images: 2846


In [ ]:
sampled = pd.read_csv(SPLIT_PATH)
required_columns = {'image_id', 'fold', 'clip'}
assert required_columns.issubset(sampled.columns), f'Missing columns: {required_columns - set(sampled.columns)}'
assert len(sampled) == 4062, f'Expected 4062 sampled images, got {len(sampled)}'
assert sampled['image_id'].is_unique
assert set(sampled['fold'].astype(int).unique()) == {0, 1, 2}
assert sampled['fold'].notna().all()

original_image_ids = {int(img['id']) for img in train_coco['images']}
assert set(sampled['image_id'].astype(int)).issubset(original_image_ids)

for fold in range(3):
    train_part = sampled.loc[sampled['fold'] != fold]
    val_part = sampled.loc[sampled['fold'] == fold]
    assert set(train_part['clip']).isdisjoint(set(val_part['clip'])), 'Video leakage detected!'
    print(
        f"Fold {fold}: "
        f"train={len(train_part)}, "
        f"val={len(val_part)}, "
        f"train clips={train_part['clip'].nunique()}, "
        f"val clips={val_part['clip'].nunique()}"
    )


assert sorted(sampled['fold'].value_counts().tolist()) == [1349, 1356, 1357]
print('Existing 3-fold split verified.')

Fold 0: train=2705, val=1357, train clips=37, val clips=19
Fold 1: train=2706, val=1356, train clips=37, val clips=19
Fold 2: train=2713, val=1349, train clips=38, val clips=18
Existing 3-fold split verified.


In [ ]:
image_info = {int(img['id']): img for img in train_coco['images']}
annotations_by_image = defaultdict(list)
for ann in train_coco['annotations']:
    annotations_by_image[int(ann['image_id'])].append(ann)


def image_path_from_info(info):
    rel = Path(info['file_name'])
    candidates = (
        DATA_ROOT / 'train' / rel,
        DATA_ROOT / 'train/images' / rel,
        DATA_ROOT / 'train/images' / rel.parent.name / rel.name,
        DATA_ROOT / rel,
    )
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f'Image not found for {info["file_name"]}; checked {candidates}')


class VisDroneRetinaDataset(Dataset):
    def __init__(self, image_ids):
        self.ids = sorted(int(x) for x in image_ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        with Image.open(image_path_from_info(image_info[image_id])) as im:
            image = im.convert('RGB')
            width, height = image.size
            image_tensor = pil_to_tensor(image).float().div_(255.0)

        boxes, labels, areas = [], [], []
        for ann in annotations_by_image[image_id]:
            if int(ann.get('iscrowd', 0)) == 1:
                continue
            category_id = int(ann['category_id'])
            if category_id not in range(1, 9):
                continue
            x, y, w, h = map(float, ann['bbox'])
            x1 = min(width, max(0.0, x))
            y1 = min(height, max(0.0, y))
            x2 = min(width, max(0.0, x + w))
            y2 = min(height, max(0.0, y + h))
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append((x1, y1, x2, y2))
            labels.append(category_id - 1)  # RetinaNet logits use 0..7
            areas.append((x2 - x1) * (y2 - y1))

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor(image_id, dtype=torch.int64),
            'area': torch.tensor(areas, dtype=torch.float32),
            'iscrowd': torch.zeros(len(labels), dtype=torch.int64),
        }
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_fold_loaders(fold):
    train_ids = sampled.loc[sampled['fold'] != fold, 'image_id']
    val_ids = sampled.loc[sampled['fold'] == fold, 'image_id']
    train_dataset = VisDroneRetinaDataset(train_ids)
    val_dataset = VisDroneRetinaDataset(val_ids)
    train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=True)
    val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn,
                        pin_memory=True)
    return train_dl, val_dl, set(int(x) for x in val_ids)

train_check, val_check, _ = make_fold_loaders(0)
images_check, targets_check = next(iter(train_check))
assert len(images_check) == BATCH_SIZE
assert all(torch.isfinite(t['boxes']).all().item() for t in targets_check)
assert all(((t['labels'] >= 0) & (t['labels'] < 8)).all().item() for t in targets_check)
print('Input tensor:', tuple(images_check[0].shape))
print('Boxes in first image:', len(targets_check[0]['boxes']))
print('Dataset / label smoke test passed.')
del train_check, val_check, images_check, targets_check

Input tensor: (3, 1071, 1904)
Boxes in first image: 44
Dataset / label smoke test passed.


## 3 — Controlled experiments
Only one factor differs from the original baseline in each trial. Original baseline: SGD lr=0.002, min/max=640/1024, focal gamma=2.0, alpha=0.25.

In [ ]:
TRIALS = {
    'resolution_768': {'optimizer':'SGD','lr':0.002,'min_size':768,'max_size':1280,'gamma':2.0},
    'adamw_2e4': {'optimizer':'AdamW','lr':0.0002,'min_size':640,'max_size':1024,'gamma':2.0},
    'focal_gamma_1p5': {'optimizer':'SGD','lr':0.002,'min_size':640,'max_size':1024,'gamma':1.5},
}
assert EPOCHS == 6 and BATCH_SIZE == 2 and SEED == 42
import json
(RESULTS_ROOT / 'trial_config.json').write_text(json.dumps(TRIALS, indent=2))
print('Candidate experiments:', *TRIALS.keys(), sep='\n - ')
print('Original baseline metrics:', BASELINE_ROOT / 'fold_0' / 'validation_metrics.csv')

Candidate experiments:
 - resolution_768
 - adamw_2e4
 - focal_gamma_1p5
Original baseline metrics: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_baseline/fold_0/validation_metrics.csv


## 4 — Model factory, optimizer and true focal-loss modification

Original torchvision RetinaNet focal classification loss uses alpha=0.25 and gamma=2.0. For `focal_gamma_1p5` we override **only** the classification-head `compute_loss`, preserving matching, ignore anchors and per-image normalization. This is a real loss modification, not just changing the optimizer. The patch is recreated when resuming from a saved checkpoint.

In [ ]:
import math
import types
from torchvision.models.detection import (
    retinanet_resnet50_fpn_v2,
    RetinaNet_ResNet50_FPN_V2_Weights,
)
from torchvision.ops import sigmoid_focal_loss


def focal_loss_with_gamma(self, targets, head_outputs, matched_idxs, gamma=1.5):
    losses = []
    for target, logits, matched in zip(targets, head_outputs['cls_logits'], matched_idxs):
        foreground = matched >= 0
        gt_target = torch.zeros_like(logits)
        gt_target[foreground, target['labels'][matched[foreground]]] = 1.0
        valid = matched != self.BETWEEN_THRESHOLDS
        loss = sigmoid_focal_loss(
            logits[valid], gt_target[valid], alpha=0.25, gamma=gamma,
            reduction='sum'
        ) / max(1, int(foreground.sum().item()))
        losses.append(loss)
    return sum(losses) / len(losses)


def build_tuning_model(cfg):
    model = retinanet_resnet50_fpn_v2(
        weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=int(cfg['min_size']), max_size=int(cfg['max_size']),
        score_thresh=0.01, nms_thresh=0.5,
        detections_per_img=300, topk_candidates=1000,
    )
    cls_head = model.head.classification_head
    old_logits = cls_head.cls_logits
    anchors_per_location = model.anchor_generator.num_anchors_per_location()
    assert len(set(anchors_per_location)) == 1
    new_logits = torch.nn.Conv2d(
        old_logits.in_channels, anchors_per_location[0] * 8,
        kernel_size=3, stride=1, padding=1,
    )
    torch.nn.init.normal_(new_logits.weight, std=0.01)
    torch.nn.init.constant_(new_logits.bias, -math.log(99.0))
    cls_head.cls_logits = new_logits
    cls_head.num_classes = 8
    if float(cfg['gamma']) != 2.0:
        gamma = float(cfg['gamma'])
        def modified_compute_loss(self, targets, head_outputs, matched_idxs):
            return focal_loss_with_gamma(self, targets, head_outputs, matched_idxs, gamma)
        cls_head.compute_loss = types.MethodType(modified_compute_loss, cls_head)
    return model.to(DEVICE)


def build_optimizer(model, cfg):
    if cfg['optimizer'] == 'SGD':
        return torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=0.0005)
    if cfg['optimizer'] == 'AdamW':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=0.0005)
    raise ValueError(cfg['optimizer'])

print('Model + true focal loss experiment ready.')

Model + true focal loss experiment ready.


## 5 — Same COCOeval as RetinaNet baseline

In [ ]:
def evaluate_model(model, val_loader, val_image_ids, fold):
    model.eval()
    gt = {
        'images': [im for im in train_coco['images'] if int(im['id']) in val_image_ids],
        'annotations': [a for a in train_coco['annotations']
                        if int(a['image_id']) in val_image_ids and 1 <= int(a['category_id']) <= 8],
        'categories': [c for c in train_coco['categories'] if 1 <= int(c['id']) <= 8],
        'info': train_coco.get('info', {}),
        'licenses': train_coco.get('licenses', []),
    }
    coco_gt = COCO()
    coco_gt.dataset = gt
    coco_gt.createIndex()

    predictions = []
    inference_times = []
    with torch.inference_mode():
        for images, targets in tqdm(val_loader, desc=f'Fold {fold}: validation', leave=False):
            images_device = [im.to(DEVICE) for im in images]
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            outputs = model(images_device)
            torch.cuda.synchronize()
            inference_times.append((time.perf_counter() - t0) / len(images))

            for out, target in zip(outputs, targets):
                image_id = int(target['image_id'].item())
                for box, conf, cls in zip(
                    out['boxes'].detach().cpu().tolist(),
                    out['scores'].detach().cpu().tolist(),
                    out['labels'].detach().cpu().tolist(),
                ):
                    if not (0 <= int(cls) < 8) or conf < 0.01:
                        continue
                    x1, y1, x2, y2 = box
                    w, h = x2 - x1, y2 - y1
                    if w <= 0 or h <= 0:
                        continue
                    predictions.append({
                        'image_id': image_id,
                        'category_id': int(cls) + 1,
                        'bbox': [float(x1), float(y1), float(w), float(h)],
                        'score': float(conf),
                    })

    if not predictions:
        print(f'Fold {fold}: zero predictions; COCO AP metrics set to 0.')
        stats = np.zeros(12, dtype=float)
    else:
        coco_dt = coco_gt.loadRes(predictions)
        evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
        evaluator.params.imgIds = sorted(val_image_ids)
        evaluator.params.catIds = list(range(1, 9))
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
        stats = evaluator.stats

    return {
        'model': 'RetinaNet ResNet50 FPN v2',
        'fold': fold,
        'epochs_completed': EPOCHS,
        'mAP50-95': float(stats[0]),
        'AP50': float(stats[1]),
        'AP75': float(stats[2]),
        'AP_small': float(stats[3]),
        'AP_medium': float(stats[4]),
        'AP_large': float(stats[5]),
        'inference_ms_per_frame': float(np.mean(inference_times) * 1000),
    }

print('COCO evaluator ready.')

COCO evaluator ready.


## 6 — Preflight: one forward / backward pass
Uses baseline-size model first to check gradient flow; this does **not** train an epoch.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
smoke_cfg = TRIALS['focal_gamma_1p5']
smoke_model = build_tuning_model(smoke_cfg)
smoke_model.train()
smoke_loader, _, _ = make_fold_loaders(0)
smoke_images, smoke_targets = next(iter(smoke_loader))
smoke_images = [x.to(DEVICE) for x in smoke_images]
smoke_targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in smoke_targets]
smoke_losses = smoke_model(smoke_images, smoke_targets)
smoke_sum = sum(smoke_losses.values())
assert torch.isfinite(smoke_sum).item()
smoke_sum.backward()
print({k: round(v.item(), 5) for k,v in smoke_losses.items()})
print('Modified focal-loss forward and backward passed.')
del smoke_model, smoke_loader, smoke_images, smoke_targets, smoke_losses, smoke_sum
gc.collect(); torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:01<00:00, 78.6MB/s]


{'classification': 1.16104, 'bbox_regression': 0.24197}
Modified focal-loss forward and backward passed.


## 7 — Train / resume a single trial on a single fold

- Each trial starts from original COCO-pretrained weights (not an existing fold checkpoint).
- A separate `latest.pt` stores model, optimizer, scheduler, GradScaler and training history **after every completed epoch**.
- Resume is allowed only for the same trial and fold.
- Final COCO metrics written only after all epochs and validation.
- Completed trials are skipped.
- Run calls individually; this avoids exhausting the GPU quota unexpectedly.

In [ ]:
def save_checkpoint_atomic(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    local = Path('/content/retina_tuning_checkpoint_tmp.pt')
    intermediate = path.with_suffix('.uploading')
    torch.save(payload, local)
    shutil.copy2(local, intermediate)
    intermediate.replace(path)


def run_trial(trial_name, fold=0):
    assert trial_name in TRIALS, f'Unknown trial: {trial_name}'
    assert fold in (0, 1, 2)
    cfg = TRIALS[trial_name].copy()
    fold_dir = RESULTS_ROOT / trial_name / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = fold_dir / 'latest.pt'
    metrics_path = fold_dir / 'validation_metrics.csv'
    history_path = fold_dir / 'history.csv'

    if metrics_path.is_file() and ckpt_path.is_file():
        ready = pd.read_csv(metrics_path)
        if (len(ready) == 1 and int(ready.iloc[0]['fold']) == fold
                and int(ready.iloc[0]['epochs_completed']) == EPOCHS):
            print(f'{trial_name}, fold {fold}: completed. Skipping.')
            return ready

    set_seed()
    gc.collect(); torch.cuda.empty_cache()
    train_loader, val_loader, val_ids = make_fold_loaders(fold)
    train_clips = set(sampled.loc[sampled['fold'] != fold, 'clip'])
    val_clips = set(sampled.loc[sampled['fold'] == fold, 'clip'])
    assert train_clips.isdisjoint(val_clips)
    print(f'{trial_name} | fold {fold}: train {len(train_loader.dataset)}, val {len(val_loader.dataset)}')
    model = build_tuning_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    history = []
    start_epoch = 0

    if ckpt_path.is_file():
        state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        assert state['trial'] == trial_name and state['cfg'] == cfg
        assert int(state['fold']) == fold and int(state['epochs']) == EPOCHS
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history = state['history']
        start_epoch = int(state['epoch']) + 1
        del state
        assert 0 <= start_epoch <= EPOCHS
        print('Resuming after', start_epoch, 'completed epochs')

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        total_loss = cls_loss = box_loss = 0.0
        t0 = time.perf_counter()
        progress = tqdm(train_loader, desc=f'{trial_name} fold {fold} epoch {epoch+1}/{EPOCHS}', leave=False)
        for images, targets in progress:
            image_gpu = [im.to(DEVICE, non_blocking=True) for im in images]
            target_gpu = [{k:v.to(DEVICE, non_blocking=True) for k,v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                losses = model(image_gpu, target_gpu)
                loss = sum(losses.values())
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f'Non-finite loss: {trial_name} fold {fold}, epoch {epoch+1}')
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item())
            cls_loss += float(losses['classification'].item())
            box_loss += float(losses['bbox_regression'].item())
            progress.set_postfix(loss=f'{loss.item():.3f}')
        scheduler.step()
        rec = {
            'trial':trial_name, 'fold':fold, 'epoch':epoch+1,
            'train_loss':total_loss/len(train_loader),
            'classification_loss':cls_loss/len(train_loader),
            'bbox_regression_loss':box_loss/len(train_loader),
            'learning_rate_next_epoch':optimizer.param_groups[0]['lr'],
            'minutes':(time.perf_counter()-t0)/60,
        }
        history.append(rec)
        save_checkpoint_atomic({
            'trial':trial_name, 'cfg':cfg, 'fold':fold, 'epochs':EPOCHS, 'seed':SEED,
            'epoch':epoch, 'model':model.state_dict(),
            'optimizer':optimizer.state_dict(), 'scheduler':scheduler.state_dict(),
            'scaler':scaler.state_dict(), 'history':history,
        }, ckpt_path)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f"{trial_name} fold {fold}: epoch {epoch+1}/{EPOCHS}, loss {rec['train_loss']:.4f}, saved")

    result = evaluate_model(model, val_loader, val_ids, fold)
    result.update({'trial':trial_name,'optimizer':cfg['optimizer'],
                   'lr':cfg['lr'],'min_size':cfg['min_size'],
                   'max_size':cfg['max_size'],'gamma':cfg['gamma'], 'batch':BATCH_SIZE})
    pd.DataFrame([result]).to_csv(metrics_path, index=False)
    print(f"COMPLETED {trial_name} fold {fold}: COCO mAP {result['mAP50-95']:.5f}")
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame([result])

print('Training/resume function defined. No model training was started.')

Training/resume function defined. No model training was started.


## 8 — Run **one candidate at a time** on fold 0
Uncomment only one call, execute, verify validation metrics, then proceed to another. Larger resolution may OOM on a smaller GPU; do not alter baseline.

In [ ]:
result_resolution = run_trial('resolution_768', fold=0)

resolution_768 | fold 0: train 2705, val 1357


resolution_768 fold 0 epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 0: epoch 1/6, loss 0.6250, saved


resolution_768 fold 0 epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 0: epoch 2/6, loss 0.4633, saved


resolution_768 fold 0 epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (5 outputs left out)

resolution_768 fold 0 epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 0: epoch 6/6, loss 0.3791, saved
creating index...
index created!


Fold 0: validation:   0%|          | 0/679 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.63s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=94.84s).
Accumulating evaluation results...
DONE (t=4.86s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.179
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.341
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.166
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.043
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.153
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.372
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.097
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.278
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.333
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [ ]:
result_adamw = run_trial('adamw_2e4', fold=0)

adamw_2e4 | fold 0: train 2705, val 1357


adamw_2e4 fold 0 epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

adamw_2e4 fold 0: epoch 1/6, loss 0.5067, saved


adamw_2e4 fold 0 epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

adamw_2e4 fold 0: epoch 2/6, loss 0.3695, saved


adamw_2e4 fold 0 epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (5 outputs left out)

adamw_2e4 fold 0 epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

adamw_2e4 fold 0: epoch 6/6, loss 0.1861, saved
creating index...
index created!


Fold 0: validation:   0%|          | 0/679 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.50s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=71.33s).
Accumulating evaluation results...
DONE (t=2.73s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.158
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.302
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.149
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.038
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.321
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.086
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.229
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.276
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [ ]:
result_focal = run_trial('focal_gamma_1p5', fold=0)

focal_gamma_1p5 | fold 0: train 2705, val 1357


focal_gamma_1p5 fold 0 epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 0: epoch 1/6, loss 0.7756, saved


focal_gamma_1p5 fold 0 epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 0: epoch 2/6, loss 0.5726, saved


focal_gamma_1p5 fold 0 epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (5 outputs left out)

focal_gamma_1p5 fold 0 epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 0: epoch 6/6, loss 0.4570, saved
creating index...
index created!


Fold 0: validation:   0%|          | 0/679 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.64s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=86.79s).
Accumulating evaluation results...
DONE (t=3.50s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.164
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.309
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.155
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.035
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.134
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.344
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.090
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.253
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.307
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

## 9 — Fold-0 comparison with existing baseline
Read results from Drive, no GPU required. Choose based on mAP50–95 primarily, also inspect AP_small and stability. Do **not** declare a winner before running trials.

In [ ]:
records = []
base_path = BASELINE_ROOT / 'fold_0' / 'validation_metrics.csv'
base = pd.read_csv(base_path)
assert len(base) == 1
records.append({'trial':'baseline_original','fold':0,**base.iloc[0].to_dict()})
for name in TRIALS:
    p = RESULTS_ROOT / name / 'fold_0' / 'validation_metrics.csv'
    if p.exists():
        frame = pd.read_csv(p)
        assert len(frame) == 1
        records.append({'trial':name,'fold':0,**frame.iloc[0].to_dict()})
comparison = pd.DataFrame(records)
cols = ['trial', 'fold', 'mAP50-95','AP50','AP75','AP_small','AP_medium','AP_large']
comparison = comparison[cols].sort_values('mAP50-95',ascending=False)
display(comparison.round(5))
comparison.to_csv(RESULTS_ROOT / 'fold0_comparison.csv',index=False)
print('Saved fold0 comparison:', RESULTS_ROOT / 'fold0_comparison.csv')

,trial,fold,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large
1,resolution_768,0,0.17924,0.34059,0.16632,0.04266,0.15300,0.37162
3,focal_gamma_1p5,0,0.16397,0.30887,0.15544,0.03512,0.13367,0.34428
0,baseline_original,0,0.15950,0.30219,0.15063,0.03457,0.13014,0.33758
2,adamw_2e4,0,0.15835,0.30222,0.14907,0.03829,0.13198,0.32057


Saved fold0 comparison: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_tuning/fold0_comparison.csv


## 10 — Verify the selected candidate on folds 1 and 2

**Only after reviewing fold-0 comparison:** set `SELECTED_TRIAL` to an actually completed trial from `TRIALS`. Do not automatically select by fold 0 and describe this as an independent test. Run folds separately to manage GPU budget.

Original baseline RetinaNet for all 3 folds is already saved and is never retrained.

In [ ]:
SELECTED_TRIAL = 'resolution_768'
assert SELECTED_TRIAL in TRIALS
print('Selected:', SELECTED_TRIAL)

Selected: resolution_768


In [ ]:
assert SELECTED_TRIAL is not None
fold1_result = run_trial(SELECTED_TRIAL, fold=1)

resolution_768 | fold 1: train 2706, val 1356


resolution_768 fold 1 epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 1: epoch 1/6, loss 0.6253, saved


resolution_768 fold 1 epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 1: epoch 2/6, loss 0.5152, saved


resolution_768 fold 1 epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (5 outputs left out)

resolution_768 fold 1 epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

resolution_768 fold 1: epoch 6/6, loss 0.4390, saved
creating index...
index created!


Fold 1: validation:   0%|          | 0/678 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.64s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=115.57s).
Accumulating evaluation results...
DONE (t=4.61s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.159
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.293
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.151
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.031
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.145
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.353
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.095
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.217
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.276
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

In [ ]:
assert SELECTED_TRIAL is not None
fold2_result = run_trial(SELECTED_TRIAL, fold=2)

resolution_768 | fold 2: train 2713, val 1349


resolution_768 fold 2 epoch 1/6:   0%|          | 0/1357 [00:00<?, ?it/s]

resolution_768 fold 2: epoch 1/6, loss 0.6252, saved


resolution_768 fold 2 epoch 2/6:   0%|          | 0/1357 [00:00<?, ?it/s]

resolution_768 fold 2: epoch 2/6, loss 0.4951, saved


resolution_768 fold 2 epoch 3/6:   0%|          | 0/1357 [00:00<?, ?it/s]

... (5 outputs left out)

resolution_768 fold 2 epoch 6/6:   0%|          | 0/1357 [00:00<?, ?it/s]

resolution_768 fold 2: epoch 6/6, loss 0.4268, saved
creating index...
index created!


Fold 2: validation:   0%|          | 0/675 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=2.12s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=98.08s).
Accumulating evaluation results...
DONE (t=4.91s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.172
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.330
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.164
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.070
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.181
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.301
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.118
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.249
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.300
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

In [ ]:
fold1_result_focal = run_trial('focal_gamma_1p5', fold=1)

focal_gamma_1p5 | fold 1: train 2706, val 1356


focal_gamma_1p5 fold 1 epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 1: epoch 1/6, loss 0.7629, saved


focal_gamma_1p5 fold 1 epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 1: epoch 2/6, loss 0.6260, saved


focal_gamma_1p5 fold 1 epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (5 outputs left out)

focal_gamma_1p5 fold 1 epoch 6/6:   0%|          | 0/1353 [00:00<?, ?it/s]

focal_gamma_1p5 fold 1: epoch 6/6, loss 0.5242, saved
creating index...
index created!


Fold 1: validation:   0%|          | 0/678 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.91s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=114.00s).
Accumulating evaluation results...
DONE (t=4.53s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.268
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.131
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.028
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.130
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.308
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.088
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.217
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.274
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=

In [ ]:
fold2_result_focal = run_trial('focal_gamma_1p5', fold=2)

focal_gamma_1p5 | fold 2: train 2713, val 1349


focal_gamma_1p5 fold 2 epoch 1/6:   0%|          | 0/1357 [00:00<?, ?it/s]

focal_gamma_1p5 fold 2: epoch 1/6, loss 0.7640, saved


focal_gamma_1p5 fold 2 epoch 2/6:   0%|          | 0/1357 [00:00<?, ?it/s]

focal_gamma_1p5 fold 2: epoch 2/6, loss 0.6105, saved


focal_gamma_1p5 fold 2 epoch 3/6:   0%|          | 0/1357 [00:00<?, ?it/s]

... (5 outputs left out)

focal_gamma_1p5 fold 2 epoch 6/6:   0%|          | 0/1357 [00:00<?, ?it/s]

focal_gamma_1p5 fold 2: epoch 6/6, loss 0.5239, saved
creating index...
index created!


Fold 2: validation:   0%|          | 0/675 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.71s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=94.95s).
Accumulating evaluation results...
DONE (t=3.60s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.149
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.290
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.035
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.145
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.288
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.108
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.227
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.279
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

## 11 — Final 3-fold baseline vs tuned comparison
Run only after selected trials on folds 0/1/2 finish. Reads files, does not train.

In [22]:
assert SELECTED_TRIAL in TRIALS, 'Select a tuned trial after fold-0 search.'
metric_cols = ['mAP50-95','AP50','AP75','AP_small','AP_medium','AP_large']
records = []
for fold in range(3):
    for variant, p in [
        ('baseline', BASELINE_ROOT / f'fold_{fold}' / 'validation_metrics.csv'),
        ('tuned', RESULTS_ROOT / SELECTED_TRIAL / f'fold_{fold}' / 'validation_metrics.csv')
    ]:
        assert p.is_file(), f'Missing: {p}'
        df = pd.read_csv(p)
        assert len(df) == 1 and int(df.iloc[0]['fold']) == fold
        records.append({'variant':variant, 'fold':fold,
                        **{k:float(df.iloc[0][k]) for k in metric_cols}})
all_results = pd.DataFrame(records).sort_values(['fold','variant'])
summary = all_results.groupby('variant')[metric_cols].agg(['mean','std'])
paired = all_results.pivot(index='fold',columns='variant',values='mAP50-95')
paired['delta_tuned_minus_baseline'] = paired['tuned'] - paired['baseline']
display(all_results.round(5))
display(summary.round(5))
display(paired.round(5))
all_results.to_csv(RESULTS_ROOT / f'{SELECTED_TRIAL}_baseline_vs_tuned_per_fold.csv',index=False)
summary.to_csv(RESULTS_ROOT / f'{SELECTED_TRIAL}_baseline_vs_tuned_summary.csv')
paired.to_csv(RESULTS_ROOT / f'{SELECTED_TRIAL}_baseline_vs_tuned_paired.csv')
print('Saved comparison CSVs on Google Drive:',RESULTS_ROOT)

,variant,fold,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large
0,baseline,0,0.15950,0.30219,0.15063,0.03457,0.13014,0.33758
1,tuned,0,0.17924,0.34059,0.16632,0.04266,0.15300,0.37162
2,baseline,1,0.14103,0.26406,0.12977,0.02730,0.13161,0.30254
3,tuned,1,0.15883,0.29297,0.15133,0.03099,0.14483,0.35330
4,baseline,2,0.14874,0.28979,0.14212,0.03443,0.15000,0.28739
5,tuned,2,0.17159,0.33012,0.16421,0.06959,0.18102,0.30122


mAP50-95              AP50              AP75          AP_small  \
             mean      std     mean      std     mean      std     mean   
variant                                                                   
baseline  0.14976  0.00928  0.28535  0.01945  0.14084  0.01048  0.03210   
tuned     0.16989  0.01031  0.32123  0.02502  0.16062  0.00811  0.04775   

                  AP_medium          AP_large           
              std      mean      std     mean      std  
variant                                                 
baseline  0.00416   0.13725  0.01107  0.30917  0.02575  
tuned     0.01979   0.15962  0.01898  0.34205  0.03652

variant,baseline,tuned,delta_tuned_minus_baseline
fold,,,
0,0.15950,0.17924,0.01974
1,0.14103,0.15883,0.01779
2,0.14874,0.17159,0.02285


Saved comparison CSVs on Google Drive: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_tuning


In [23]:
import pandas as pd

variants = {
    "baseline": BASELINE_ROOT,
    "resolution_768": RESULTS_ROOT / "resolution_768",
    "focal_gamma_1p5": RESULTS_ROOT / "focal_gamma_1p5",
}

metrics = ["mAP50-95", "AP50", "AP75",
           "AP_small", "AP_medium", "AP_large"]

rows = []

for name, root in variants.items():
    for fold in range(3):
        path = root / f"fold_{fold}" / "validation_metrics.csv"
        assert path.exists(), f"Missing file: {path}"

        df = pd.read_csv(path)
        assert len(df) == 1

        rows.append({
            "model": name,
            "fold": fold,
            **{m: float(df.iloc[0][m]) for m in metrics}
        })

results = pd.DataFrame(rows)
summary = results.groupby("model")[metrics].agg(["mean", "std"])

display(results.round(5))
display(summary.round(5))

results.to_csv(
    RESULTS_ROOT / "all_tuning_3fold_results.csv",
    index=False
)
summary.to_csv(
    RESULTS_ROOT / "all_tuning_3fold_summary.csv"
)

print("Saved all RetinaNet results to Google Drive.")

,model,fold,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large
0,baseline,0,0.15950,0.30219,0.15063,0.03457,0.13014,0.33758
1,baseline,1,0.14103,0.26406,0.12977,0.02730,0.13161,0.30254
2,baseline,2,0.14874,0.28979,0.14212,0.03443,0.15000,0.28739
3,resolution_768,0,0.17924,0.34059,0.16632,0.04266,0.15300,0.37162
4,resolution_768,1,0.15883,0.29297,0.15133,0.03099,0.14483,0.35330
5,resolution_768,2,0.17159,0.33012,0.16421,0.06959,0.18102,0.30122
6,focal_gamma_1p5,0,0.16397,0.30887,0.15544,0.03512,0.13367,0.34428
7,focal_gamma_1p5,1,0.14216,0.26750,0.13071,0.02802,0.13035,0.30781
8,focal_gamma_1p5,2,0.14875,0.28989,0.14197,0.03517,0.14455,0.28787


mAP50-95              AP50              AP75           \
                    mean      std     mean      std     mean      std   
model                                                                   
baseline         0.14976  0.00928  0.28535  0.01945  0.14084  0.01048   
focal_gamma_1p5  0.15163  0.01118  0.28875  0.02071  0.14271  0.01238   
resolution_768   0.16989  0.01031  0.32123  0.02502  0.16062  0.00811   

                AP_small          AP_medium          AP_large           
                    mean      std      mean      std     mean      std  
model                                                                   
baseline         0.03210  0.00416   0.13725  0.01107  0.30917  0.02575  
focal_gamma_1p5  0.03277  0.00411   0.13619  0.00743  0.31332  0.02860  
resolution_768   0.04775  0.01979   0.15962  0.01898  0.34205  0.03652

Saved all RetinaNet results to Google Drive.


In [24]:
from pathlib import Path

for trial in ("resolution_768", "focal_gamma_1p5"):
    for fold in range(3):
        checkpoint = (
            RESULTS_ROOT / trial / f"fold_{fold}" / "latest.pt"
        )
        print(
            trial,
            fold,
            "OK" if checkpoint.is_file() else "MISSING"
        )

resolution_768 0 OK
resolution_768 1 OK
resolution_768 2 OK
focal_gamma_1p5 0 OK
focal_gamma_1p5 1 OK
focal_gamma_1p5 2 OK


## Notes for interpretation

- Original baseline and new candidates have the same sampled 3-fold video split, `batch=2`, `epochs=6` and COCOeval protocol.
- RetinaNet baseline has pretrained backbone/other heads and a newly initialized 8-class classification logit layer; each candidate is independently initialized the same way.
- Focal gamma modification changes classification loss only, leaving anchor matching and localization loss unchanged.
- Fold 0 is reused for candidate selection, so the final 3-fold estimate is not a fully independent post-selection test. Preserve original holdout video set for final checking.
- Higher resolution may help small objects but can increase GPU memory and latency. Compare `AP_small` and elapsed training time, not just mAP.
- This is architecture tuning for the HW. Kaggle leaderboard improvements must be checked separately using the competition test format.